# Combinatorial background ROI-blinded workflow

Canonical notebook for the DUNE ND-LAr 2x2 MCP double-blip accidental background estimate. Configure paths below, run the lightweight blinding step, then inspect summary and pair-angle shapes. Heavy production pair sampling should be run from `scripts/blinding.py` on batch resources.

In [ ]:
from pathlib import Path
import json
import subprocess

import h5py
import matplotlib.pyplot as plt
import numpy as np

INPUT_H5 = Path("example_input.h5")  # replace with collaborator data file
OUTPUT_DIR = Path("outputs/roi_blinded_signal")
MODE = "signal"
N_PAIRS = 100_000
MIN_DISTANCE = 0.0
CENTER_ZX = 0.0
CENTER_ZY = 0.0
THETA_RADIUS = 0.10


## Run the canonical blinding script

This cell is intentionally guarded so the notebook parses without data. Set `RUN_BLINDING = True` after updating `INPUT_H5`.

In [ ]:
RUN_BLINDING = False

if RUN_BLINDING:
    cmd = [
        "python", "scripts/blinding.py",
        "--input", str(INPUT_H5),
        "--output-dir", str(OUTPUT_DIR),
        "--mode", MODE,
        "--n-pairs", str(N_PAIRS),
        "--min-distance", str(MIN_DISTANCE),
        "--center-zx", str(CENTER_ZX),
        "--center-zy", str(CENTER_ZY),
        "--theta-radius", str(THETA_RADIUS),
    ]
    subprocess.run(cmd, check=True)
else:
    print("Set RUN_BLINDING=True after configuring INPUT_H5 to produce outputs.")


## Inspect normalization summary

In [ ]:
summary_path = OUTPUT_DIR / "summary.json"
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    display(summary)
else:
    print(f"No summary found yet: {summary_path}")


## Plot synthetic pair angular shape outside the blinded ROI

In [ ]:
pairs_path = OUTPUT_DIR / "blinded_singles_and_pairs.h5"
if pairs_path.exists():
    with h5py.File(pairs_path, "r") as handle:
        pairs = handle["synthetic_pairs"][:]
    fig, ax = plt.subplots(figsize=(6, 5))
    hist = ax.hist2d(pairs["theta_zx"], pairs["theta_zy"], bins=80)
    fig.colorbar(hist[3], ax=ax, label="synthetic pairs")
    ax.set_xlabel(r"$\theta_{zx}$ [rad]")
    ax.set_ylabel(r"$\theta_{zy}$ [rad]")
    ax.set_title("Singles-driven double-blip shape after canonical selection")
    plt.show()
else:
    print(f"No pair file found yet: {pairs_path}")
